# Bird Identification using CV - Part 4: Refinement

**ITAI 1378  |  Midterm  |  2026**

**Group 8**

**Author:** Stuart Fairchild | Kalen Foster | Ranveer Chand


---

## 4a.  Custom detection model

Using the Kaggle 2000 birds dataset, we have trained a custom model to draw bounding boxes around birds for detection. This section evaluates the model's performance against our previous best performer, YOLO11-Large

In [1]:
%pip install -q ultralytics

from ultralytics import YOLO, SAM
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

print("Setup complete. Ready to detect and segment.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 50.3 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Setup complete. Ready to detect and segment.


---

## 4b.  Load images from Google Drive

Connect to Google Drive to load test image directory.

To use this without changes, add images to Google Drive under

`/Colab Notebooks/ITAI1378/midterm/images`

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
base_path = '/content/drive/MyDrive/Colab Notebooks/ITAI1378/ITAI1378-midterm'
images_path = '/content/drive/MyDrive/Colab Notebooks/ITAI1378/ITAI1378-midterm/images'

for filename in os.listdir(images_path):
  if filename.endswith((".png", ".jpg")):
    file_path = os.path.join(images_path, filename)
    print(f"Current file: {filename}")
    # Uncomment below if you want to load all images for viewing
    # img = Image.open(file_path)
    # plt.figure(figsize=(8, 10))
    # plt.imshow(img)
    # plt.axis("off")
    # plt.show()

Current file: 20260711_110012851062.jpg
Current file: 20260711_110607700821.jpg
Current file: 20260711_110614124173.jpg
Current file: 20260711_110623390691.jpg
Current file: 20260711_114504335390.jpg
Current file: 20260711_114510046678.jpg
Current file: 20260711_114518126954.jpg
Current file: 20260711_114652178971.jpg
Current file: 20260711_114656013898.jpg
Current file: 20260711_122042601055.jpg
Current file: 20260711_122047486200.jpg
Current file: 20260711_122057774137.jpg
Current file: 20260711_124529347193.jpg
Current file: 20260711_125021249691.jpg
Current file: 20260711_125032098571.jpg
Current file: 20260711_125043301951.jpg
Current file: 20260711_131423814430.png
Current file: 20260711_131457505871.png
Current file: 20260711_163118393903.png
Current file: 20260711_163140593762.png
Current file: 20260711_163151124444.png
Current file: 20260711_163202648535.png
Current file: 20260711_163215955185.png
Current file: 20260711_163224382030.png
Current file: 20260711_195811947191.png


## 6. Running detection with bird_detect_2000

bird_detect_2000 is our custom model for bird detection. Trained from the dataset found at:

https://www.kaggle.com/datasets/gpiosenka/birdies?resource=download

In [7]:
model = "bird_detect_2000"
detector = YOLO(f"models/bird_detect_2000.pt")
print(f"{model} loaded.")

detected_path = f"{base_path}/detected_{model}"
for filename in os.listdir(images_path):
  if filename.endswith((".png", ".jpg")):
    file_path = os.path.join(images_path, filename)
    print(f"Current file: {filename}")
    results = detector(file_path)
    annotated = results[0].plot()
    annotated_rgb = annotated[..., ::-1]
    plt.figure(figsize=(10, 12))
    plt.imshow(annotated_rgb)
    plt.axis("off")
    plt.title(f"{model} detection results: {filename}")
    output_file = f"{detected_path}/{filename}"
    print(f"Output detected file: {output_file}")
    plt.savefig(output_file)
    plt.show()

Output hidden; open in https://colab.research.google.com to view.

## 7. Running detection with YOLO11 Large


In [8]:
model = "YOLO11Large"
detector = YOLO("yolo11l.pt")
print(f"{model} loaded.")

detected_path = f"{base_path}/detected_{model}"
for filename in os.listdir(images_path):
  if filename.endswith((".png", ".jpg")):
    file_path = os.path.join(images_path, filename)
    print(f"Current file: {filename}")
    results = detector.predict(
        source=file_path,
        conf=0.12,           # Set confidence threshold lower
        classes=[14]        # 14 is the COCO index for 'bird'
    )
    annotated = results[0].plot()
    annotated_rgb = annotated[..., ::-1]
    plt.figure(figsize=(10, 12))
    plt.imshow(annotated_rgb)
    plt.axis("off")
    plt.title(f"{model} detection results: {filename}")
    output_file = f"{detected_path}/{filename}"
    print(f"Output detected file: {output_file}")
    plt.savefig(output_file)
    plt.show()


Output hidden; open in https://colab.research.google.com to view.

## Compare YOLO11Large and bird_detect_2000 side by side

In [9]:
import matplotlib.pyplot as plt
from PIL import Image
import os

model1 = "YOLO11Large"
model2 = "bird_detect_2000"
image1_path = f"{base_path}/detected_{model1}"
image2_path = f"{base_path}/detected_{model2}"
output_path = f"{base_path}/comp_YOLO11L_bird_detect_2000"

# Create the output directory if it doesn't exist
os.makedirs(output_path, exist_ok=True)

for filename in os.listdir(image1_path):
  if filename.endswith((".png", ".jpg")):
    full_path1 = os.path.join(image1_path, filename)
    full_path2 = os.path.join(image2_path, filename)

    # Check if both files exist before trying to open them
    if os.path.exists(full_path1) and os.path.exists(full_path2):
      img1 = Image.open(full_path1)
      img2 = Image.open(full_path2)

      fig, axes = plt.subplots(1, 2, figsize=(25, 15)) # Increased figsize for larger images
      axes[0].imshow(img1)
      axes[0].axis('off')
      axes[0].set_title(f"{model1} detection results: {filename}")
      axes[1].imshow(img2)
      axes[1].axis('off')  # Hide axes
      axes[1].set_title(f"{model2} detection results: {filename}")

      output_file = os.path.join(output_path, filename)
      print(f"Output comparison file: {output_file}")
      plt.savefig(output_file)

      plt.tight_layout()
      plt.show()
    else:
      print(f"Skipping {filename}: one or both image files not found.")

Output hidden; open in https://colab.research.google.com to view.

## Conclusions

The YOLO Large models performed best in this initial proof of concept.


## Compare YOLO11nano and bird_detect_2000 side by side

In [10]:
import matplotlib.pyplot as plt
from PIL import Image
import os

model1 = "YOLO11nano"
model2 = "bird_detect_2000"
image1_path = f"{base_path}/detected_{model1}"
image2_path = f"{base_path}/detected_{model2}"
output_path = f"{base_path}/comp_YOLO11L_bird_detect_2000"

# Create the output directory if it doesn't exist
os.makedirs(output_path, exist_ok=True)

for filename in os.listdir(image1_path):
  if filename.endswith((".png", ".jpg")):
    full_path1 = os.path.join(image1_path, filename)
    full_path2 = os.path.join(image2_path, filename)

    # Check if both files exist before trying to open them
    if os.path.exists(full_path1) and os.path.exists(full_path2):
      img1 = Image.open(full_path1)
      img2 = Image.open(full_path2)

      fig, axes = plt.subplots(1, 2, figsize=(25, 15)) # Increased figsize for larger images
      axes[0].imshow(img1)
      axes[0].axis('off')
      axes[0].set_title(f"{model1} detection results: {filename}")
      axes[1].imshow(img2)
      axes[1].axis('off')  # Hide axes
      axes[1].set_title(f"{model2} detection results: {filename}")

      output_file = os.path.join(output_path, filename)
      print(f"Output comparison file: {output_file}")
      plt.savefig(output_file)

      plt.tight_layout()
      plt.show()
    else:
      print(f"Skipping {filename}: one or both image files not found.")

Output hidden; open in https://colab.research.google.com to view.

## 6. Running detection with bird_detect_2001

bird_detect_2001 is the second iteration of our custom model for bird detection. 2001 uses YOLO11L as the base model for training.

Trained from the dataset found at:

https://www.kaggle.com/datasets/gpiosenka/birdies?resource=download

In [7]:
model = "bird_detect_2001"
detector = YOLO(f"models/bird_detect_2001.pt")
print(f"{model} loaded.")

detected_path = f"{base_path}/detected_{model}"
for filename in os.listdir(images_path):
  if filename.endswith((".png", ".jpg")):
    file_path = os.path.join(images_path, filename)
    print(f"Current file: {filename}")
    results = detector(file_path)
    annotated = results[0].plot()
    annotated_rgb = annotated[..., ::-1]
    plt.figure(figsize=(10, 12))
    plt.imshow(annotated_rgb)
    plt.axis("off")
    plt.title(f"{model} detection results: {filename}")
    output_file = f"{detected_path}/{filename}"
    print(f"Output detected file: {output_file}")
    plt.savefig(output_file)
    plt.show()

Output hidden; open in https://colab.research.google.com to view.